# Initial ENTSOE exploration
Checking the structure of the ingested XML file and how it coverts to a table format using Pandas as a single day file should be very small.

In [0]:
# entsoe-py is a third-party client for the ENTSO-E API — not in the Databricks
# runtime, so it needs installing. It wraps the same API and returns pandas directly,
# so I can compare it with my manual requests + xml.etree parsing.
# %pip install is notebook-scoped (only this session, doesn't touch the shared cluster)
# and restarts Python, so it must be the first cell.
%pip install entsoe-py

In [0]:
from entsoe import EntsoePandasClient
import pandas as pd

client = EntsoePandasClient(api_key=dbutils.secrets.get("default2", "gabriela-entsoe-token"))
start = pd.Timestamp("20260710", tz="Europe/Warsaw")
end   = pd.Timestamp("20260711", tz="Europe/Warsaw")
prices = client.query_day_ahead_prices("PL", start=start, end=end)
print(prices.head())

In [0]:
# Importing data directly from the Entsoe API

import requests

# Secret added to our shared secret store
token = dbutils.secrets.get(scope="default2", key="gabriela-entsoe-token")

params = {
    "securityToken": token,
    "documentType": "A44",
    "in_Domain":  "10YPL-AREA-----S",
    "out_Domain": "10YPL-AREA-----S",
    "periodStart": "202607100000",
    "periodEnd":   "202607110000",
}

r = requests.get("https://web-api.tp.entsoe.eu/api", params=params, timeout=60)
print("status:", r.status_code)
print(r.text[:2000])

In [0]:
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
import pandas as pd

root = ET.fromstring(r.text)
ns = {"ns": root.tag.split("}")[0].strip("{")} # Unpacking ENTSOE-E namespace

rows = []
for ts in root.findall("ns:TimeSeries", ns):
    currency = ts.findtext("ns:currency_Unit.name", namespaces = ns)
    unit = ts.findtext("ns:price_Measure_Unit.name", namespaces = ns)
    for period in ts.findall("ns:Period", ns):
        start = period.findtext("ns:timeInterval/ns:start", namespaces=ns)
        res   = period.findtext("ns:resolution", namespaces=ns)          # PT15M or PT60M
        start_dt = datetime.strptime(start, "%Y-%m-%dT%H:%MZ").replace(tzinfo=timezone.utc)
        step = timedelta(minutes=15 if res == "PT15M" else 60)
        for point in period.findall("ns:Point", ns):
            pos   = int(point.findtext("ns:position", namespaces=ns))
            price = float(point.findtext("ns:price.amount", namespaces=ns))
            rows.append((start_dt + step*(pos-1), price, currency, unit))

pdf = pd.DataFrame(rows, columns=["timestamp_utc", "price", "currency", "unit"])
print(pdf.head(12))
print("Number of rows:", len(pdf))